In [1]:
!pip install -r ../requirements.txt 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.2/283.2 MB 1.9 MB/s eta 0:00:0000:0100:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 2.8 MB/s eta 0:00:0000:0100:01


In [ ]:
# brush will be here ~/brush/target/release/brush_app 

In [85]:
# 1. Path declaration

import uuid
from pathlib import Path
import shutil

# User options from frontend
USE_DOWNSAMPLE = False
USE_REMBG = True

# Video file path (e.g. uploaded from frontend)
video_path = Path("/home/jovyan/mira3d/uploads/shiva.MOV")  

# Setup UUID run directory
run_uuid = str(uuid.uuid4())
run_dir = Path("../data") / run_uuid
frames_dir = run_dir / "frames"
masked_dir = run_dir / "masked_frames"
colmap_dir = run_dir / "colmap"
brush_dir = run_dir / "brush"
run_dir.mkdir(parents=True)
frames_dir.mkdir()
if USE_REMBG:
    masked_dir.mkdir()
colmap_dir.mkdir()
brush_dir.mkdir()

# Copy video to run folder
input_video_path = run_dir / "input.mp4"
shutil.copy(video_path, input_video_path)

print(" Setup done:", run_uuid)


 Setup done: f28c2eb0-0978-426b-9a18-ecc504553a30


In [86]:
# 2. Extract frames from video

import subprocess

fps = "2"
subprocess.run([
    "ffmpeg", "-i", str(input_video_path),
    "-vf", f"fps={fps}",
    str(frames_dir / "%04d.jpg")
], check=True)

print("Frames extracted:", len(list(frames_dir.glob('*.jpg'))))


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

Frames extracted: 116


frame=  116 fps= 17 q=24.8 Lsize=N/A time=00:00:58.00 bitrate=N/A speed=8.33x    
video:3694kB audio:0kB subtitle:0kB other streams:0kB global headers:0kB muxing overhead: unknown


In [87]:
# 3. Optional downsampling with ImageMagick

if USE_DOWNSAMPLE:
    downsample_dir = run_dir / "frames_2"
    downsample_dir.mkdir(exist_ok=True)

    # Resize all images to 50%
    !magick mogrify -path "{downsample_dir}" -resize 50%% "{frames_dir}/*.jpg"
    
    # Use downsampled frames for rest of pipeline
    frames_dir = downsample_dir

    print(" Images downsampled to 50%.")


In [88]:
# 4. Optional background removal using rembg


if USE_REMBG:
    from rembg import remove, new_session
    from PIL import Image

    session = new_session("u2net", providers=["CUDAExecutionProvider"])
    for img_file in frames_dir.glob("*.jpg"):
        img = Image.open(img_file)
        out = remove(img, session=session)
        out.save(masked_dir / f"{img_file.stem}.png")  # Save as .png

    # Use masked frames for the rest of the pipeline
    frames_dir = masked_dir
    print("Backgrounds removed using rembg.")


Backgrounds removed using rembg.


In [89]:
# Feature extraction: during this step COLMAP detects keypoints in each image using the
# SIFT (Scale-Invariant Feature Transform ) algorithm it will use gpu as out colmap with built with cuda
# --database_path: specifies where the features and matches will be stored(a sql lite db)
# -- iamgepath: the path to the folder containing our dataset images 


DB_PATH = colmap_dir / "database.db"

subprocess.run([
    "colmap", "feature_extractor",
    "--database_path", str(DB_PATH),
    "--image_path", str(frames_dir),
    "--ImageReader.single_camera", "1",
    "--ImageReader.camera_model", "PINHOLE",
    "--SiftExtraction.use_gpu", "1"
], check=True)

print(" COLMAP feature extraction complete.")
    

I0607 22:52:40.800063 34368 misc.cc:44] 
Feature extraction
I0607 22:52:40.801064 34381 sift.cc:721] Creating SIFT GPU feature extractor
I0607 22:52:40.961971 34382 feature_extraction.cc:259] Processed file [1/116]
I0607 22:52:40.961997 34382 feature_extraction.cc:262]   Name:            0001.png
I0607 22:52:40.962000 34382 feature_extraction.cc:271]   Dimensions:      1080 x 1920
I0607 22:52:40.962002 34382 feature_extraction.cc:274]   Camera:          #1 - PINHOLE
I0607 22:52:40.962005 34382 feature_extraction.cc:277]   Focal Length:    2304.00px
I0607 22:52:40.962013 34382 feature_extraction.cc:281]   Features:        2160
I0607 22:52:41.301412 34382 feature_extraction.cc:259] Processed file [2/116]
I0607 22:52:41.301481 34382 feature_extraction.cc:262]   Name:            0002.png
I0607 22:52:41.301491 34382 feature_extraction.cc:271]   Dimensions:      1080 x 1920
I0607 22:52:41.301493 34382 feature_extraction.cc:274]   Camera:          #1 - PINHOLE
I0607 22:52:41.301497 34382 feat

 COLMAP feature extraction complete.


In [90]:
# 6. COLMAP Matching The exhaustive matcher compares each image with every other image to 
# find matching points between them, which is crucial for the 3D reconstruction process

subprocess.run([
    "colmap", "exhaustive_matcher",
    "--database_path", str(DB_PATH),
    "--SiftMatching.use_gpu", "1"
], check=True)

print("✅ COLMAP matching complete.")


I0607 22:52:58.044765 34456 misc.cc:44] 
Feature matching
I0607 22:52:58.045238 34457 sift.cc:1428] Creating SIFT GPU feature matcher
I0607 22:52:58.146646 34456 pairing.cc:172] Generating exhaustive image pairs...
I0607 22:52:58.146658 34456 pairing.cc:205] Matching block [1/3, 1/3]
I0607 22:52:58.376504 34456 feature_matching.cc:47] in 0.230s
I0607 22:52:58.376525 34456 pairing.cc:205] Matching block [1/3, 2/3]
I0607 22:52:58.559500 34456 feature_matching.cc:47] in 0.183s
I0607 22:52:58.559521 34456 pairing.cc:205] Matching block [1/3, 3/3]
I0607 22:52:58.587028 34456 feature_matching.cc:47] in 0.028s
I0607 22:52:58.587050 34456 pairing.cc:205] Matching block [2/3, 1/3]
I0607 22:52:58.770947 34456 feature_matching.cc:47] in 0.184s
I0607 22:52:58.770968 34456 pairing.cc:205] Matching block [2/3, 2/3]
I0607 22:52:58.908980 34456 feature_matching.cc:47] in 0.138s
I0607 22:52:58.909000 34456 pairing.cc:205] Matching block [2/3, 3/3]
I0607 22:52:58.922003 34456 feature_matching.cc:47] in 

✅ COLMAP matching complete.


I0607 22:52:59.122167 34456 feature_matching.cc:47] in 0.016s
I0607 22:52:59.122186 34456 timer.cc:91] Elapsed time: 0.018 [minutes]


In [91]:
# 7. COLMAP Mapping (Sparse reconstruction) Structure from motion (SfM) is the process 
# of estimating the 3-D structure of a scene from a set of 2-D images. SfM is used in 
# many applications, such as 3-D scanning, augmented reality, and visual simultaneous localization and mapping (vSLAM)

SPARSE_PATH = colmap_dir / "sparse"
SPARSE_PATH.mkdir(exist_ok=True)

subprocess.run([
    "colmap", "mapper",
    "--database_path", str(DB_PATH),
    "--image_path", str(frames_dir),
    "--output_path", str(SPARSE_PATH)
], check=True)

print("✅ COLMAP sparse reconstruction complete.")


I0607 22:53:03.361635 34493 incremental_pipeline.cc:253] Loading database
I0607 22:53:03.362668 34493 database_cache.cc:66] Loading rigs...
I0607 22:53:03.362680 34493 database_cache.cc:76]  1 in 0.000s
I0607 22:53:03.362684 34493 database_cache.cc:84] Loading cameras...
I0607 22:53:03.362695 34493 database_cache.cc:102]  1 in 0.000s
I0607 22:53:03.362699 34493 database_cache.cc:110] Loading frames...
I0607 22:53:03.363301 34493 database_cache.cc:127]  116 in 0.001s
I0607 22:53:03.363305 34493 database_cache.cc:135] Loading matches...
I0607 22:53:03.365084 34493 database_cache.cc:140]  735 in 0.002s
I0607 22:53:03.365090 34493 database_cache.cc:156] Loading images...
I0607 22:53:03.370662 34493 database_cache.cc:238]  116 in 0.006s (connected 115)
I0607 22:53:03.370672 34493 database_cache.cc:249] Building correspondence graph...
I0607 22:53:03.377822 34493 database_cache.cc:276]  in 0.007s (ignored 0)
I0607 22:53:03.377899 34493 timer.cc:91] Elapsed time: 0.000 [minutes]
I0607 22:53:0

✅ COLMAP sparse reconstruction complete.


I0607 22:53:22.198218 34493 incremental_pipeline.cc:571] Discarding reconstruction due to insufficient size
I0607 22:53:22.199405 34493 incremental_pipeline.cc:299] Finding good initial image pair
I0607 22:53:22.201413 34493 incremental_pipeline.cc:303] => No good initial image pair found.
I0607 22:53:22.201418 34493 incremental_pipeline.cc:538] Discarding reconstruction due to no initial pair
I0607 22:53:22.201503 34493 timer.cc:91] Elapsed time: 0.314 [minutes]


In [92]:
# Copy images into sparse/0 so Brush can find them
# 8. Copy images into sparse/0 so Brush can find them (PNG or JPG depending on input)
import shutil

image_output_dir = SPARSE_PATH / "0"
for img in frames_dir.glob("*.[jp][pn]g"): 
    shutil.copy(img, image_output_dir)


In [93]:
# 9. Run Brush
BRUSH_BIN = "/home/jovyan/mira3d/brush/target/release/brush_app"
export_name = f"{run_uuid}.ply"
colmap_model_path = SPARSE_PATH / "0"

subprocess.run([
    BRUSH_BIN,
    str(colmap_model_path),
    "--total-steps", "30000",
    "--export-path", str(brush_dir),
    "--export-name", export_name,
    "--export-every", "30000"
], check=True)

print(f"✅ Brush reconstruction complete. Output saved to: {brush_dir / export_name}")

error: XDG_RUNTIME_DIR not set in the environment.
error: XDG_RUNTIME_DIR not set in the environment.
error: XDG_RUNTIME_DIR not set in the environment.
error: XDG_RUNTIME_DIR not set in the environment.


✅ Brush reconstruction complete. Output saved to: ../data/f28c2eb0-0978-426b-9a18-ecc504553a30/brush/f28c2eb0-0978-426b-9a18-ecc504553a30.ply
